# Beginner Tutorial 7: When Things Go Wrong (Error Handling)

## What You Will Learn

- Why distributed errors are harder than local ones
- Common failure modes (OOM, timeout, network)
- Retry strategies with exponential backoff
- Idempotency (safe to retry)
- Partial success (keep good results, handle failures separately)

## Prerequisites

- Completed notebooks 01, 06
- `pip install scalable`

In [ ]:
import os
import tempfile
import time
import random

project_dir = tempfile.mkdtemp(prefix="scalable-beginner-07-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## 💡 Key Concept: Why Distributed Errors Are Harder

On your laptop: function raises exception → you see traceback → you fix it.

In distributed systems, NEW failure modes exist:
- **Network failure** — result computed but never delivered
- **Partial failure** — 3 of 4 workers succeed, 1 fails
- **Timing issues** — can't tell "failed" from "slow"
- **Cascading failure** — one failure triggers others

## 💡 Key Concept: Idempotency

An operation is **idempotent** if running it multiple times = running it once.

✅ Idempotent (safe to retry): `x = 5`, reading a file, `f(x)` for pure functions

❌ NOT idempotent (dangerous to retry): `x += 1`, sending email, charging credit card

**For retries to be safe, your tasks must be idempotent!**

## 💡 Key Concept: Exponential Backoff

Wait progressively longer between retries:
- Attempt 1: fail → wait 1s
- Attempt 2: fail → wait 2s  
- Attempt 3: fail → wait 4s
- Attempt 4: fail → wait 8s

Why? If failure is caused by overload, retrying immediately makes it worse.

In [ ]:
# Write manifest
manifest_content = """\
version: 1
project:
  name: error-handling-demo

targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none

components:
  analysis:
    cpus: 1
    memory: 1G

tasks:
  run_analysis:
    component: analysis
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest_content)
print("Manifest ready.")

In [ ]:
from scalable import ScalableSession

def sometimes_fails(task_id: int) -> dict:
    """A function that randomly fails 30% of the time.
    
    This simulates transient failures (network issues, timeouts, etc.)
    """
    time.sleep(0.2)
    if random.random() < 0.3:  # 30% chance of failure
        raise RuntimeError(f"Transient failure on task {task_id}!")
    return {"task_id": task_id, "result": task_id * 10}

print("Function defined: fails ~30% of the time (simulating transient errors)")

## Step 1: Handling Partial Success

**Partial success** = some tasks succeed, others fail.
Don't throw away 95% good results because 5% failed!

In [ ]:
# Run with partial success handling
session = ScalableSession.from_manifest("./scalable.yaml", target="local")

# Submit tasks
random.seed(42)  # For reproducibility
futures = [session.submit(sometimes_fails, i, task="run_analysis") for i in range(20)]

# Gather with error isolation (don't let one failure crash everything)
results = []
failures = []

for i, future in enumerate(futures):
    try:
        result = future.result()  # Get individual result
        results.append(result)
    except Exception as e:
        failures.append({"task_id": i, "error": str(e)})

print(f"Results: {len(results)} succeeded, {len(failures)} failed")
print(f"\n✅ Successes: {[r['task_id'] for r in results]}")
print(f"❌ Failures: {[f['task_id'] for f in failures]}")
print(f"\n💡 We kept {len(results)} good results despite {len(failures)} failures!")

session.close()

## 🤔 Think About It

Without partial success handling, ONE failure would crash the entire `gather()`.
You'd lose ALL results — even the ones that succeeded.

With individual error handling, you keep everything that worked and can
retry just the failures.

## 💡 Key Concept: Fault Tolerance

**Fault tolerance** = ability to continue operating despite failures.

Levels:
1. **Crash** — any failure stops everything (fragile)
2. **Detect** — failures caught and reported clearly
3. **Retry** — transient failures automatically retried
4. **Partial success** — good results preserved
5. **Self-healing** — system recovers automatically

Scalable provides levels 2–5 depending on configuration.

## 📖 Vocabulary Summary

| Term | Definition |
|------|------------|
| Fault Tolerance | Continuing to operate despite failures |
| Transient Failure | Temporary error that resolves on retry |
| Permanent Failure | Error that won't fix by retrying |
| Idempotency | Safe to run multiple times (same result) |
| Exponential Backoff | Progressively longer waits between retries |
| Partial Success | Some tasks succeed, others fail |
| Exception | Python's error signaling (raise/try/except) |
| Graceful Degradation | Reducing service rather than crashing |

In [ ]:
# Cleanup
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print(f"Cleaned up {project_dir}")